In [2]:
#import packages 
from pprint import pprint
import tensorflow as tf
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts

I0000 00:00:1780754597.431203   39107 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet = tts.dataTensorLoading(testSet)


In [4]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet) :
    print(name[0])
    print(name[1])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet) :
    print(name[0])
    print(name[1])
    print("============")

data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4.nii
data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4_@37-label.nii
data/CTA nii/1125852-maryamd...hghan/1125852-maryamd...hghan_B_&_N_CTA_20220113202635_4.nii
data/CTA nii/1125852-maryamd...hghan/1125852-maryamd...hghan_B_&_N_CTA_20220113202635_4_@47-label.nii
data/CTA nii/1087276-HAJARROKHI/1087276-HAJARROKHI_CTA_20200829091700_4.nii
data/CTA nii/1087276-HAJARROKHI/1087276-HAJARROKHI_CTA_20200829091700_4_@41-label.nii
data/CTA nii/924918--ZOHREHVAHIDI/924918--ZOHREHVAHIDI_Brain_-CTA_20151213001224_4.nii
data/CTA nii/924918--ZOHREHVAHIDI/924918--ZOHREHVAHIDI_Brain_-CTA_20151213001224_4_@21-label.nii
data/CTA nii/979393-MAHDIZAREHEI/979393-MAHDIZAREHEI_Brain_-CTA_20170630001056_4.nii
data/CTA nii/979393-MAHDIZAREHEI/979393-MAHDIZAREHEI_Brain_-CTA_20170630001056_4_@16-label.nii
data/CTA nii/607664-GHOLAMRE...YBANI/607664-GHOLAMRE...YBANI_Brain_-CTA_20180107140922_4.nii
dat

In [22]:
# pipeline configuring

geo      = utl.randomGeo(p=1)
crop     = utl.volume_crop((128 , 128 , 128))
windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=1 ,
    p_ww=1
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64])
def rimg(imgPath , labelPath) :
    return utl.read_img(imgPath , labelPath)
def read_img(img , label) :
    imglbl = rimg(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    img , label = geo.flip(
        img , 
        label
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



In [ ]:
# ram optimization ==> rerunning with more number of parallel task and higher buffer size , decreasing in ram allocation maybe occur
dataloaderCache = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )
    .cache("myCache")
)

dataloaderPerEpoch = (
    dataloaderCache
    .shuffle(buffer_size=10)
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .batch(batch_size=2)
)

In [ ]:
# step by step running for managing ram allocation

In [6]:
# model compilation

tf.config.list_physical_devices('CPU')

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [ ]:
cnt=0

for data in dataloaderPerEpoch.take(10) :
    cnt+=1
    print(cnt)